# Практика 4: Triton и FlashAttention


## 1. Установка и импорты

In [ ]:
!pip list | grep "triton"

triton                                   3.6.0


In [ ]:
# !pip install triton


In [ ]:
import math
import torch
import pandas as pd

import triton
import triton.language as tl

print(f"torch: {torch.__version__}")


torch: 2.10.0+cu128


## Сложение векторов на triton

Каждый program instance в тритон обрабатывает блок элементов. `tl.arange` позволяет получить локальные смещения внутри блока, а `mask` защищает последний неполный блок от выхода за границу массива.

Отдельно стоит рассмотреть `tl.constexpr`. `tl.constexpr` помечает аргумент кернела как константу, известную на этапе компиляции. Это означает, что переменная статична. Это необходимо для некоторых инструкций тритона, например для `tl.arange`, но ввод разных значений может привести к повторной компиляции кернела.



In [ ]:

@triton.jit
def vector_add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
  pid = tl.program_id(axis=0)
  offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
  mask = offsets < n_elements

  x = tl.load(x_ptr + offsets, mask=mask, other=0)
  y = tl.load(y_ptr + offsets, mask=mask, other=0)

  tl.store(out_ptr + offsets, x + y, mask=mask)


In [ ]:
def vector_add(x: torch.Tensor, y: torch.Tensor, block_size: int = 1024) -> torch.Tensor:
  # Сначала необходимо проверить, что вход корректный
  if x.shape != y.shape:
    raise ValueError(f"x and y must have same shape")

  # оба входных тензора должны лежать на куде
  if not x.is_cuda or not y.is_cuda:
    raise ValueError("x and y must be CUDA tensors")

  # тензоры должны распологаться в памяти непрерывно
  x = x.contiguous()
  y = y.contiguous()

  # аллоцируем память под выходной тензор
  out = torch.empty_like(x)
  n_elements = x.numel()

  # нужно задать сетку program instance'ов:
  # размерность тензора может не делиться нацело на размер блока
  # в таком случае округляем вверх
  grid = (triton.cdiv(n_elements, block_size), )
  vector_add_kernel[grid](x, y, out, n_elements, block_size)
  return out

In [ ]:
n = 1_000_003
dtype = torch.float32

x = torch.randn(n, device="cuda", dtype=dtype)
y = torch.randn(n, device="cuda", dtype=dtype)
out = vector_add(x, y)
torch.testing.assert_close(out, x + y)

## Наивный аттеншн на torch
При наивной реализации мы материализуем матрицу внимания размерности `seq_len x seq_len`.


In [ ]:
def naive_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
  if q.ndim != 4 or k.ndim != 4 or v.ndim != 4:
      raise ValueError("q, k and v must have shape [batch, heads, seq_len, head_dim]")
  if q.shape != k.shape or q.shape != v.shape:
      raise ValueError(f"q, k and v must have the same shape, got {q.shape}, {k.shape}, {v.shape}")

  head_dim = q.shape[-1]
  scale = 1.0 / math.sqrt(head_dim)
  scores = torch.matmul(q.float(), k.float().transpose(-2, -1)) * scale
  attn = torch.softmax(scores, dim=-1)
  out = torch.matmul(attn, v.float())
  return out.to(dtype=q.dtype)


In [ ]:
q = torch.randn(2, 3, 16, 32)
k = torch.randn_like(q)
v = torch.randn_like(q)
out = naive_attention(q, k, v)
assert out.shape == q.shape
print(out.shape)


torch.Size([2, 3, 16, 32])


## FlashAttention (только forward)

Для блока строк `Q_i` последовательно проходим по блокам `(K_t, V_t)`.

Для каждой строки блока неообходимо поддерживать:

- `m` - текущий максимум scores;
- `l` - текущая сумма экспонент относительно `m`;
- `acc` - ненормированная сумма `sum exp(S - m) V`.

А для блока строк надо поддерживать соответствующие вектора.

После обработки всех блоков записываем `O_i = acc / l`.


Обсудим аргументы

Сначала мы передаем указатели на наши входные тензоры `q_ptr`, `k_ptr`, `v_ptr` и выходной тензор `out_ptr`

Они имеют размерность `[b, h, n, d]` - сначала количество батчей, потом количество голов, следом длина последовательности и в конце - размерность эмбеддинга.

Эти тензоры хранятся в памяти в виде 1D-колбаски чисел. Чтобы получить доступ к нужному элементу, необходимо также передать либо размерности тензоров, либо страйды. Страйды говорят, насколько элементов необходимо сдвинуться вдоль этой 1D-колбаски при изменении индекса по определенной размерности:

```
stride_d = 1
stride_n = d
stride_h = n * d
stride_b = h * n * d

x.shape = [2, 3, 4, 5]
x.stride()  # (60, 20, 5, 1)
```

То есть чтобы сдвинуться в таком тензоре вдоль `axis=0`, необходимо сдвинуться на 60 элементов вдоль плоского представления тензора.


In [ ]:
@triton.jit
def flash_attention_fwd_kernel(
    q_ptr, k_ptr, v_ptr, out_ptr,
    stride_qb, stride_qh, stride_qn, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_on, stride_od,
    n_heads: tl.constexpr,
    head_dim: tl.constexpr,
    seq_len: tl.constexpr,
    scale: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr
):
# Получаем нужные pid
  pid_m = tl.program_id(axis=0)
  pid_bh = tl.program_id(axis=1)

# Ищем оффсеты
  off_b = pid_bh // n_heads
  off_h = pid_bh - off_b * n_heads
  off_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
  off_n = tl.arange(0, BLOCK_N)
  off_d = tl.arange(0, BLOCK_D)

# Вычисляем маску по данным
  mask_d = off_d < head_dim

# Находим блок данных, который мы будем обрабатывать
  q_base = q_ptr + off_b * stride_qb + off_h * stride_qh
  k_base = k_ptr + off_b * stride_kb + off_h * stride_kh
  v_base = v_ptr + off_b * stride_vb + off_h * stride_vh
  o_base = out_ptr + off_b * stride_ob + off_h * stride_oh

# Загружаем блок строк Q_i
  q = tl.load(
      q_base + off_m[:, None] * stride_qn + off_d[None, :] * stride_qd,
      mask=(off_m[:, None] < seq_len) & mask_d[None, :],
      other=0.0
  )

# Инициализируем наши "бегущие" статистики
  m_i = tl.full((BLOCK_M,), -float("inf"), tl.float32)
  l_i = tl.zeros((BLOCK_M,), tl.float32)
  acc = tl.zeros((BLOCK_M, BLOCK_D), tl.float32)

# Цикл по парам (K_t, V_t)
  for start_n in range(0, seq_len, BLOCK_N):
    # Вычисляем оффсет для текущего блока
    off_k = start_n + off_n

    # Загружаем в память тайлы K_t и V_t
    # Тритон сам поймет, что нужно будет грузить их в SMEM
    k = tl.load(
        k_base + off_k[:, None] * stride_kn + off_d[None, :] * stride_kd,
        mask=(off_k[:, None] < seq_len) & mask_d[None, :],
        other=0.0,
    )
    v = tl.load(
        v_base + off_k[:, None] * stride_vn + off_d[None, :] * stride_vd,
        mask=(off_k[:, None] < seq_len) & mask_d[None, :],
        other=0.0,
    )

    # Вычислим блок оценок матрицы внимания
    scores = tl.dot(q, tl.trans(k)) * scale

    # Маскируем те элементы блока, которые выходят за границу последовательности
    scores = tl.where(
        (off_m[:, None] < seq_len) & (off_k[None, :] < seq_len),
        scores,
        -float("inf"),
    )
    # ВАЖНО: это пока еще не маскированое внимание

    # Обновляем статистики
    m_new = tl.maximum(m_i, tl.max(scores, axis=1))
    p = tl.exp(scores - m_new[:, None])
    alpha = tl.exp(m_i - m_new)
    l_new = alpha * l_i + tl.sum(p, axis=1)

    acc = acc * alpha[:, None] + tl.dot(p.to(tl.float32), v.to(tl.float32))
    m_i = m_new
    l_i = l_new

    # Считаем строку результата O_i
  out = acc / l_i[:, None]

  tl.store(
      o_base + off_m[:, None] * stride_on + off_d[None, :] * stride_od,
      out,
      mask=(off_m[:, None] < seq_len) & mask_d[None, :]
  )




In [ ]:
def flash_attention_forward(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    block_m: int = 16,
    block_n: int = 64,
) -> torch.Tensor:
    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4:
        raise ValueError("q, k and v must have shape [batch, heads, seq_len, head_dim]")
    if q.shape != k.shape or q.shape != v.shape:
        raise ValueError(f"q, k and v must have the same shape, got {q.shape}, {k.shape}, {v.shape}")

    batch, n_heads, seq_len, head_dim = q.shape

    q = q.contiguous()
    k = k.contiguous()
    v = v.contiguous()
    out = torch.empty_like(q)

    block_d = triton.next_power_of_2(head_dim)
    scale = 1.0 / math.sqrt(head_dim)
    grid = (triton.cdiv(seq_len, block_m), batch * n_heads)

    flash_attention_fwd_kernel[grid](
        q, k, v, out,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        out.stride(0), out.stride(1), out.stride(2), out.stride(3),
        n_heads, head_dim, seq_len,
        scale,
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_D=block_d,
        num_warps=4,
    )
    return out


### `torch.autograd.Function`


In [ ]:
class FlashAttentionFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, block_m: int = 16, block_n: int = 64):
        ctx.save_for_backward(q, k, v)
        ctx.block_m = block_m
        ctx.block_n = block_n
        return flash_attention_forward(q, k, v, block_m=block_m, block_n=block_n)

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor):
        q, k, v = ctx.saved_tensors

        # Здесь должен быть бэквард
        ...
        return ... # flash_attention_backward()


def flash_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    block_m: int = 64,
    block_n: int = 64,
) -> torch.Tensor:
    return FlashAttentionFunction.apply(q, k, v, block_m, block_n)


In [ ]:
def check_flash_attention_forward(
    batch: int = 2,
    heads: int = 4,
    seq_len: int = 128,
    head_dim: int = 64,
    dtype: torch.dtype = torch.float16,
) -> None:
    torch.manual_seed(0)
    q = torch.randn(batch, heads, seq_len, head_dim, device="cuda", dtype=dtype)
    k = torch.randn_like(q)
    v = torch.randn_like(q)

    expected = naive_attention(q, k, v)
    actual = flash_attention(q, k, v)
    torch.testing.assert_close(actual, expected, rtol=3e-2, atol=3e-2)
    print("ok", (actual - expected).abs().max().item())


check_flash_attention_forward()


ok 0.0001220703125


## Бенчмарки


In [ ]:
def benchmark_attention_time(
    seq_lengths=(128, 256, 512, 1024, 2048),
    block_m_values=(16, 32, 64),
    block_n: int = 64,
    batch: int = 1,
    heads: int = 8,
    head_dim: int = 64,
    dtype: torch.dtype = torch.float16,
    warmup: int = 25,
    rep: int = 100,
):
    rows = []
    for seq_len in seq_lengths:
        torch.manual_seed(0)
        q = torch.randn(batch, heads, seq_len, head_dim, device="cuda", dtype=dtype)
        k = torch.randn_like(q)
        v = torch.randn_like(q)

        torch.cuda.synchronize()
        torch_ms = triton.testing.do_bench(
            lambda: naive_attention(q, k, v),
            warmup=warmup,
            rep=rep,
        )

        for block_m in block_m_values:
            torch.cuda.synchronize()
            triton_ms = triton.testing.do_bench(
                lambda: flash_attention(q, k, v, block_m=block_m, block_n=block_n),
                warmup=warmup,
                rep=rep,
            )
            row = {
                "seq_len": seq_len,
                "batch": batch,
                "heads": heads,
                "head_dim": head_dim,
                "dtype": str(dtype).replace("torch.", ""),
                "block_m": block_m,
                "block_n": block_n,
                "torch_ms": torch_ms,
                "triton_ms": triton_ms,
                "speedup": torch_ms / triton_ms,
            }
            rows.append(row)
            print(row)

    return pd.DataFrame(rows)


time_results = benchmark_attention_time()
time_results



{'seq_len': 128, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 16, 'block_n': 64, 'torch_ms': 0.11729189791740516, 'triton_ms': 0.09984246030067787, 'speedup': 1.1747697078395094}
{'seq_len': 128, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 32, 'block_n': 64, 'torch_ms': 0.11729189791740516, 'triton_ms': 0.0403107469789235, 'speedup': 2.9096929902770423}
{'seq_len': 128, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 64, 'block_n': 64, 'torch_ms': 0.11729189791740516, 'triton_ms': 0.05335999979842, 'speedup': 2.1981240322433093}
{'seq_len': 256, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 16, 'block_n': 64, 'torch_ms': 0.09490920025855303, 'triton_ms': 0.15075957971183876, 'speedup': 0.6295400958264946}
{'seq_len': 256, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 32, 'block_n': 64, 'torch_ms': 0.09490920025855303, 'triton_ms': 0.12909087061117858, 'speedup':

,seq_len,batch,heads,head_dim,dtype,block_m,block_n,torch_ms,triton_ms,speedup
0,128,1,8,64,float16,16,64,0.117292,0.099842,1.174770
1,128,1,8,64,float16,32,64,0.117292,0.040311,2.909693
2,128,1,8,64,float16,64,64,0.117292,0.053360,2.198124
3,256,1,8,64,float16,16,64,0.094909,0.150760,0.629540
4,256,1,8,64,float16,32,64,0.094909,0.129091,0.735212
5,256,1,8,64,float16,64,64,0.094909,0.096170,0.986886
6,512,1,8,64,float16,16,64,0.356698,0.545544,0.653839
7,512,1,8,64,float16,32,64,0.356698,0.482941,0.738596
8,512,1,8,64,float16,64,64,0.356698,0.374082,0.953530
9,1024,1,8,64,float16,16,64,1.259535,2.113808,0.595861


In [ ]:
def benchmark_attention_memory(
    seq_lengths=(128, 256, 512, 1024, 2048),
    block_m_values=(16, 32, 64),
    block_n: int = 64,
    batch: int = 1,
    heads: int = 8,
    head_dim: int = 64,
    dtype: torch.dtype = torch.float16,
):
    rows = []
    for seq_len in seq_lengths:
        torch.manual_seed(0)
        q = torch.randn(batch, heads, seq_len, head_dim, device="cuda", dtype=dtype)
        k = torch.randn_like(q)
        v = torch.randn_like(q)

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        baseline_memory = torch.cuda.memory_allocated()
        torch.cuda.synchronize()

        out = naive_attention(q, k, v)
        torch.cuda.synchronize()
        torch_peak_extra_mb = (torch.cuda.max_memory_allocated() - baseline_memory) / 1024**2
        del out

        for block_m in block_m_values:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            baseline_memory = torch.cuda.memory_allocated()
            torch.cuda.synchronize()

            out = flash_attention(q, k, v, block_m=block_m, block_n=block_n)
            torch.cuda.synchronize()
            triton_peak_extra_mb = (torch.cuda.max_memory_allocated() - baseline_memory) / 1024**2
            del out

            row = {
                "seq_len": seq_len,
                "batch": batch,
                "heads": heads,
                "head_dim": head_dim,
                "dtype": str(dtype).replace("torch.", ""),
                "block_m": block_m,
                "block_n": block_n,
                "torch_peak_extra_mb": torch_peak_extra_mb,
                "triton_peak_extra_mb": triton_peak_extra_mb,
                "memory_ratio": torch_peak_extra_mb / triton_peak_extra_mb,
            }
            rows.append(row)
            print(row)

    return pd.DataFrame(rows)


memory_results = benchmark_attention_memory()
memory_results


{'seq_len': 128, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 16, 'block_n': 64, 'torch_peak_extra_mb': 1.5, 'triton_peak_extra_mb': 0.125, 'memory_ratio': 12.0}
{'seq_len': 128, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 32, 'block_n': 64, 'torch_peak_extra_mb': 1.5, 'triton_peak_extra_mb': 0.125, 'memory_ratio': 12.0}
{'seq_len': 128, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 64, 'block_n': 64, 'torch_peak_extra_mb': 1.5, 'triton_peak_extra_mb': 0.125, 'memory_ratio': 12.0}
{'seq_len': 256, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 16, 'block_n': 64, 'torch_peak_extra_mb': 5.2451171875, 'triton_peak_extra_mb': 0.25, 'memory_ratio': 20.98046875}
{'seq_len': 256, 'batch': 1, 'heads': 8, 'head_dim': 64, 'dtype': 'float16', 'block_m': 32, 'block_n': 64, 'torch_peak_extra_mb': 5.2451171875, 'triton_peak_extra_mb': 0.25, 'memory_ratio': 20.98046875}
{'seq_len': 256, 'batch': 1, '

,seq_len,batch,heads,head_dim,dtype,block_m,block_n,torch_peak_extra_mb,triton_peak_extra_mb,memory_ratio
0,128,1,8,64,float16,16,64,1.500000,0.125,12.000000
1,128,1,8,64,float16,32,64,1.500000,0.125,12.000000
2,128,1,8,64,float16,64,64,1.500000,0.125,12.000000
3,256,1,8,64,float16,16,64,5.245117,0.250,20.980469
4,256,1,8,64,float16,32,64,5.245117,0.250,20.980469
5,256,1,8,64,float16,64,64,5.245117,0.250,20.980469
6,512,1,8,64,float16,16,64,18.000000,0.500,36.000000
7,512,1,8,64,float16,32,64,18.000000,0.500,36.000000
8,512,1,8,64,float16,64,64,18.000000,0.500,36.000000
9,1024,1,8,64,float16,16,64,68.245117,1.000,68.245117
